# BeltWatch: train on Colab

Runs the data pipeline, the baselines, and the U-Net on a Colab GPU. See `docs/training.md`.

**Runtime → Change runtime type → GPU** first.

## Persist outputs on Google Drive
Checkpoints and the MLflow database survive a disconnect.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/beltwatch-runs"
!mkdir -p {WORK}

## Set up

In [ ]:
!git clone https://github.com/professor3333/beltwatch.git /content/beltwatch
%cd /content/beltwatch
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os

os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
!uv sync --extra cu126
!uv run python -c "import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')"

## Build the dataset (about 7.5 GB download)

In [ ]:
!uv run dvc repro
!cat data/splits/zerowaste-f-splits.json | head -40

## Classical baselines (val only)

In [ ]:
!uv run python -m beltwatch.baselines.random_forest --out {WORK}/models/random-forest
!uv run python scripts/evaluate.py --model all-background --split val --out-dir {WORK}/reports
!uv run python scripts/evaluate.py --model random-forest --model-dir {WORK}/models/random-forest --split val --out-dir {WORK}/reports

## U-Net: write a Drive-backed config, time it, then train

In [ ]:
import yaml

with open("configs/unet.yaml") as fh:
    c = yaml.safe_load(fh)
c["output_dir"] = f"{WORK}/models/unet-resnet18-ce"
c["tracking"]["mlflow_uri"] = f"sqlite:///{WORK}/mlflow.db"
c["runtime"]["num_workers"] = 2
with open("/content/unet-colab.yaml", "w") as fh:
    yaml.safe_dump(c, fh)

In [ ]:
!uv run python scripts/train.py --config /content/unet-colab.yaml --max-steps 300 --max-val-images 50

In [ ]:
# Full run. After a disconnect, add: --resume {WORK}/models/unet-resnet18-ce/latest.pt
!uv run python scripts/train.py --config /content/unet-colab.yaml

In [ ]:
!uv run python scripts/evaluate.py --model unet --model-dir {WORK}/models/unet-resnet18-ce --split val --out-dir {WORK}/reports

## SegFormer-B0 challenger (matched configuration)

In [ ]:
import yaml

with open("configs/segformer_b0.yaml") as fh:
    c = yaml.safe_load(fh)
c["output_dir"] = f"{WORK}/models/segformer-b0-ce"
c["tracking"]["mlflow_uri"] = f"sqlite:///{WORK}/mlflow.db"
c["runtime"]["num_workers"] = 2
with open("/content/segformer-colab.yaml", "w") as fh:
    yaml.safe_dump(c, fh)

In [ ]:
!uv run python scripts/train.py --config /content/segformer-colab.yaml
!uv run python scripts/evaluate.py --model segformer --model-dir {WORK}/models/segformer-b0-ce --split val --out-dir {WORK}/reports

## Keep the dataset version
Commit `data/manifests/`, `data/splits/` and `dvc.lock` from this run through a PR (or download them and commit locally).

In [ ]:
!git status --short data dvc.lock